## Load data

In [2]:
import numpy as np
import pandas as pd
import json
from dataclasses import dataclass

def load_json_dict(filename):
    # Open a file in read mode ('r')
    with open(f'{filename}.json', 'r') as json_file:
        # Use json.load() to read the data and convert it back to a dictionary
        return json.load(json_file)
    
def get_model_metadata(model_id):
    return model_metadatas[str(model_id)]

def get_model_data(model_id):
    return model_datas[str(model_id)]

def get_model_diet_data(model_id):
    return model_diet_datas[str(model_id)]

model_metadatas = load_json_dict('model_metadatas')
model_datas = load_json_dict('model_datas')
model_diet_datas = load_json_dict('model_diet_datas')

In [ ]:
@dataclass
class SpeciesGroup:
    group_name: str
    group_seq: int
    model_number: int
    model_name: str
    model_country: str
    model_year: str
    lme: int
    tl: float
    biomass: float
    pb: float
    qb: float
    ee: float
    M0b: float
    flow_to_det: float
    gross_efficiency: float
    prop_unassimilated_food: float
    respiration: float
    biomass_accum: float
    biomass_accum_rate: float
    immigration: float
    emigration: float
    export: float
    trophic_info: str
    detritus_import: float
    taxons_included: list[dict] 

    @classmethod
    def from_dict(cls, data: dict):
        """Factory method to create an instance from a dictionary."""
        return cls(**data)

    def __str__(self):
        return f"[{self.model_number}] '{self.model_name}' {self.model_country} ({self.model_year}) -> {self.group_seq}: {self.group_name} | tl: {self.tl}, taxons included: {[self.taxons_included[i]['taxon_name'] + ' (' + self.taxons_included[i]['AphiaID'] + ')' for i in range(len(self.taxons_included))]}"
    
    def to_df_row(self):
        dct = self.__dict__.copy()
        dct.pop("taxons_included")
        return pd.DataFrame(dct)

def decoder(dct):
    return SpeciesGroup.from_dict(dct)

def read_json_SpeciesGroup_list(filename):
    # Open the JSON file and load the data
    with open(f'{filename}.json', 'r') as f:
        # Use json.load and specify the object_hook
        return json.load(f, object_hook=decoder)
    
species_groups = read_json_SpeciesGroup_list("SpeciesGroups")

### functions to get information on specific model

In [4]:
def get_model_groups_data(model_number):
    def to_df_row(c: SpeciesGroup):
        dct = c.__dict__.copy()
        dct.pop("taxons_included")
        return pd.DataFrame([dct])

    df_rows = [to_df_row(species_groups[i]) for i in range(len(species_groups)) if species_groups[i].model_number == model_number]
    df = pd.concat(df_rows, ignore_index=True)
    df = df.replace("-9999", np.nan).replace(-9999, np.nan)

    df = df.rename(columns={  # change names
        'export': 'catch',
        'prop_unassimilated_food': 'gs',
        'gross_efficiency': 'ge'
        })
    
    df['p'] = df['pb'] * df['biomass']  # production
    df['q'] = df['qb'] * df['biomass']  # consumption
    df['M0'] = df['p'] * (1-df['ee'])  # other mortality
    df['net_migration'] = df['emigration'] - df['immigration']  # net migration
    
      # production*EE = catch + predation + biomass_accum + net_migration:
    df['predation'] = df['p'] * df['ee'] - (df['catch'] + df['biomass_accum'] + df['net_migration'])
    df['egestion'] = df['q'] * df['gs']  # gs

    df['flow_to_det'] = df['egestion'] + df['M0']

    cols_to_return = ['group_name', 'trophic_info', 'tl', 'ge', 'ee', 'catch', 
        'biomass', 'pb', 'qb', 'p', 'q', 'predation', 'M0', 'gs', 'egestion', 'respiration', 'biomass_accum', 'emigration', 'immigration', 'net_migration',
         'flow_to_det', 'detritus_import',        
        ]

    return df.set_index('group_seq')[cols_to_return]

def get_seq2name(model_number):
    model = get_model_diet_data(model_number)
    if not isinstance(model, dict):
        return None
    
    groups = model.get('group', {})
    if len(groups) == 0:
        return None
    
    return {int(g["group_seq"]): g["group_name"] for g in groups}

def get_DC(model_number):
    model = get_model_diet_data(model_number)
    if not isinstance(model, dict):
        return None
    
    groups = model.get('group', {})
    if len(groups) == 0:
        return None
    
    n = len(groups)
    
    DC_dict = {}
    detritus_fate_dict = {}

    for g in groups:
        diet_descr = g.get('diet_descr', {})
        diet_descr = diet_descr if diet_descr else {}
        diet = diet_descr.get('diet', None)
        if not diet:
            DC_dict[int(g['group_seq'])] = {int(g['group_seq']): None}
            detritus_fate_dict[int(g['group_seq'])] = {int(g['group_seq']): None}
        else:
            diet = diet if isinstance(diet, list) else [diet]
            DC_dict[int(g['group_seq'])] = {int(d['prey_seq']): float(d['proportion']) for d in diet}
            detritus_fate_dict[int(g['group_seq'])] = {int(d['prey_seq']): float(d['detritus_fate']) for d in diet}
    
    # add missing columns:
    DC = pd.DataFrame.from_dict(DC_dict, orient='index').fillna(0)
    for g in groups:
        if int(g['group_seq']) not in DC.columns:
            DC[int(g['group_seq'])] = 0

    DC = DC.sort_index().sort_index(axis=1)

    detritus_fate = pd.DataFrame.from_dict(detritus_fate_dict, orient='index').fillna(0)
    detritus_fate = detritus_fate.sort_index().sort_index(axis=1)

    return DC, detritus_fate

# Implementation

In [5]:
model_number = 227  # Iceland

# general data:
seq2name = get_seq2name(model_number)
groups_data = get_model_groups_data(model_number).fillna(0)

# DC and detritus_fate matrices:
DC, det_fate = get_DC(model_number)

Z = DC.mul(groups_data['q'], axis='index')
DET_seq = list(groups_data[groups_data['trophic_info'] == 'DET'].index.values)
if len(DET_seq) == 1:
    Z.loc[DET_seq[0], :] = groups_data['flow_to_det']
elif len(DET_seq) == 2:  # det_date acts as a switch between DET groups
    Z.loc[DET_seq[0], :] = groups_data['flow_to_det'] * (DC * (1-det_fate)).sum(axis=1)
    Z.loc[DET_seq[1], :] = groups_data['flow_to_det'] * (DC * (det_fate)).sum(axis=1)
else:
    raise Exception('too many detritus groups(?)')

production = groups_data['p'].copy()

# combine PP to single row:
PP_seq_list = sorted(groups_data.index[groups_data['trophic_info'] == 'PP'].values)
PP_seq = PP_seq_list[-1]
seq_to_drop = PP_seq_list[:-1]
if len(PP_seq_list) > 1:

    production.loc[PP_seq] += production.loc[seq_to_drop].sum()
    production = production.drop(index=seq_to_drop)

    Z.loc[:, PP_seq] += Z.loc[:, seq_to_drop].sum(axis=1)
    Z = Z.drop(index=seq_to_drop, columns=seq_to_drop)

# combine part of DET that is PP into PP row:
DET_seq = groups_data[groups_data['trophic_info'] == 'DET'].index.values[0]
Z_without_DET = Z.copy()
percent_of_det_that_is_PP = Z_without_DET.loc[DET_seq, PP_seq] / Z_without_DET.loc[DET_seq, :].sum()  # 98%
Z_without_DET.loc[:, PP_seq] += percent_of_det_that_is_PP * Z_without_DET.loc[:, DET_seq]
Z_without_DET = Z_without_DET.drop(index=DET_seq, columns=DET_seq)

# production of living compartments:
new_index = Z_without_DET.index
P = production[new_index].copy()
ee = groups_data.loc[new_index, "ee"]
non_PP = groups_data['trophic_info'] == 'Regular'
P[non_PP] = P[non_PP].mul(ee[non_PP])   # P*EE = (export + predation + growth + net_migration), without M0
# EE = (export + predation + growth + net_migration) / (export + predation + growth + net_migration + M0)
P = P.sort_index(ascending=False)

# production-normalized transaction matrix:
A = Z_without_DET.T.sort_index(ascending=False).sort_index(axis=1, ascending=False) / P
A = A.rename(index=seq2name, columns=seq2name)

# production requirement matrix:
seq_to_drop = P.index[P == 0]
A = A.drop(columns=seq_to_drop, index=seq_to_drop)
L = pd.DataFrame(np.linalg.inv((np.identity(A.shape[0]) - A)), index=A.index, columns=A.columns)

L

,Phytoplankton,Zooplankton,Other Fish,Benthos,Molluscs,Northern Shrimp,Nephrops,Other Pelagics,Capelin,Herring,...,Greenland Halibut,Redfish,Saithe,Haddock,Juvenile Cod,Adult Cod,Seabirds,Pinnipeds,Baleen whales,Toothed whales
Phytoplankton,1.0,3.57432,55.776089,21.959211,146.944117,45.799731,45.117829,66.262510,17.457306,17.356974,...,598.543183,5.782683e+01,1.294161e+02,9.598161e+01,73.132026,203.561990,4435.944206,45547.052511,4929.048340,350307.199244
Zooplankton,0.0,1.00000,8.720015,0.000000,16.822295,2.894737,2.631579,7.726950,4.736842,4.709618,...,70.191237,7.621788e+00,2.193759e+01,5.938570e+00,4.612672,25.300662,633.395302,5387.600280,717.094056,42276.630708
Other Fish,0.0,0.00000,1.074024,0.000000,1.683056,0.000000,0.000000,0.138745,0.000000,0.000000,...,4.388946,2.476570e-01,1.506263e+00,1.285835e-01,0.263795,1.216444,11.627126,315.606664,28.784070,2516.021891
Benthos,0.0,0.00000,1.120167,1.000000,3.858830,1.578947,1.578947,1.759545,0.000000,0.000000,...,15.603387,1.385514e+00,2.283310e+00,3.384323e+00,2.544750,5.068848,95.688767,1176.298495,105.848862,8902.318808
Molluscs,0.0,0.00000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.400221,5.485752e-19,-1.172908e-20,-7.141261e-21,0.000000,0.037513,0.000000,61.807871,4.776354,656.839054
Northern Shrimp,0.0,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,1.312793,8.227147e-02,2.008450e-02,1.950284e-01,0.526316,0.779348,0.000000,100.795178,8.816625,615.256361
Nephrops,0.0,0.00000,0.000000,0.000000,0.008903,0.000000,1.000000,0.000000,0.000000,0.000000,...,0.008027,1.362139e-20,-2.912387e-22,-1.773210e-22,0.000000,0.000876,0.000000,1.706375,1.313530,31.710923
Other Pelagics,0.0,0.00000,0.696406,0.000000,1.091307,0.000000,0.000000,1.305284,0.000000,0.000000,...,3.874144,2.885112e-01,9.904810e-01,8.708577e-02,0.171047,1.201584,11.995948,259.298257,24.063325,2004.792103
Capelin,0.0,0.00000,0.005730,0.000000,0.008980,0.000000,0.000000,0.000740,1.000000,0.000000,...,3.560681,1.592161e-01,1.573959e+00,5.397352e-01,0.001407,1.813784,103.582251,301.603513,25.104365,2273.152871
Herring,0.0,0.00000,0.013312,0.000000,0.020860,0.000000,0.000000,0.010146,0.000000,1.000000,...,0.074976,3.956510e-03,2.402800e-02,1.619435e-03,0.003270,0.047447,1.469014,54.300563,10.348078,353.916118


In [11]:
L.loc[[seq2name[PP_seq]], :]

,Phytoplankton,Zooplankton,Other Fish,Benthos,Molluscs,Northern Shrimp,Nephrops,Other Pelagics,Capelin,Herring,...,Greenland Halibut,Redfish,Saithe,Haddock,Juvenile Cod,Adult Cod,Seabirds,Pinnipeds,Baleen whales,Toothed whales
Phytoplankton,1.0,3.57432,55.776089,21.959211,146.944117,45.799731,45.117829,66.26251,17.457306,17.356974,...,598.543183,57.826826,129.416118,95.981615,73.132026,203.56199,4435.944206,45547.052511,4929.04834,350307.199244


In [1]:
from SPPR_2015 import *

SPPR, seq2name, DC, Z, P, A, L, groups_data, PP_seq, DET_seq, new_index = SPPR_2015(model_number=227)  # Iceland 1950

SPPR.rename(index=seq2name)

Phytoplankton             1.000000
Zooplankton               3.574320
Other Fish               55.776089
Benthos                  21.959211
Molluscs                146.944117
Northern Shrimp          45.799731
Nephrops                 45.117829
Other Pelagics           66.262510
Capelin                  17.457306
Herring                  17.356974
Other Dem. Fish         230.735659
Other Flatfish          171.268168
Greenland Halibut       598.543183
Redfish                  57.826826
Saithe                  129.416118
Haddock                  95.981615
Juvenile Cod             73.132026
Adult Cod               203.561990
Seabirds               4435.944206
Pinnipeds             45547.052511
Baleen whales          4929.048340
Toothed whales       350307.199244
Name: 23, dtype: float64

In [8]:
# P.sort_index(ascending=False).sort_index(ascending=False, axis=1).rename(index=seq2name, columns=seq2name)
P.sort_index(ascending=False).rename(index=seq2name)

Phytoplankton        17243.892518
Zooplankton            101.798436
Other Fish               5.192891
Benthos                 19.406477
Molluscs                 0.831379
Northern Shrimp          1.742974
Nephrops                 0.082723
Other Pelagics           4.355170
Capelin                  5.189411
Herring                  1.014292
Other Dem. Fish          0.680416
Other Flatfish           0.437523
Greenland Halibut        0.313843
Redfish                  1.170128
Saithe                   0.387632
Haddock                  0.374264
Juvenile Cod             0.382883
Adult Cod                1.172191
Seabirds                 0.008827
Pinnipeds                0.006611
Baleen whales            0.046074
Toothed whales           0.001575
Name: p, dtype: float64

### In the Article, They used TE instead of GE (should use GE):

In [5]:
catch = groups_data['catch']
predation = Z_without_DET.sum(axis=0).sort_index(ascending=False)
growth = groups_data['biomass_accum']
net_migration = groups_data['net_migration']
M0 = groups_data['M0']
# p = catch + predation + growth + net_migration + M0

q = Z_without_DET.sum(axis=1).sort_index(ascending=False)
ge_times_ee = groups_data['p'] * groups_data['ee'] / q   # This is what was used in 2015

df = pd.DataFrame({
    'ge': groups_data['ge'].round(decimals=6),
    'ge_calc': ((catch + predation + growth + net_migration + M0) / q).round(decimals=6),
    'ge*ee': ge_times_ee.round(decimals=6),
    'te_calc': ((catch + predation + growth + net_migration) / q).round(decimals=6),
    'ee': groups_data['ee'],
    'ee_calc': ((catch + predation + growth + net_migration) / (catch + predation + growth + net_migration + M0)).round(decimals=6)
})
df.sort_index(ascending=False).rename(index=seq2name)

,ge,ge_calc,ge*ee,te_calc,ee,ee_calc
Detritus,0.000000,NaN,NaN,NaN,0.017924,NaN
Phytoplankton,0.000000,inf,inf,inf,0.400000,0.591069
Benthic producers,0.000000,NaN,NaN,NaN,0.007927,NaN
Zooplankton,0.294498,0.294498,0.279773,0.279773,0.950000,0.950000
Other Fish,0.200000,0.200000,0.190000,0.190000,0.950000,0.950000
Benthos,0.090000,0.091078,0.045539,0.045539,0.500000,0.500000
Molluscs,0.150000,0.150599,0.112949,0.112949,0.750000,0.750000
Northern Shrimp,0.200000,0.200339,0.190322,0.190322,0.950000,0.950000
Nephrops,0.200000,0.200509,0.190483,0.190483,0.950000,0.950000
Other Pelagics,0.150000,0.150000,0.142500,0.142500,0.950000,0.950000


In [111]:
model_number = 227

seq2name = get_seq2name(model_number)
name2seq = {v: k for k, v in seq2name.items()}
model_groups_data = get_model_groups_data(model_number).fillna(0)
DC, det_fate = get_DC(model_number)
Z = DC.mul(model_groups_data['q'], axis='index')
DC = DC.sort_index(ascending=False).rename(index=seq2name)
Z = Z.sort_index(ascending=False).sort_index(axis=1, ascending=False).rename(index=seq2name, columns=seq2name)
model_groups_data = model_groups_data.sort_index(ascending=False).rename(index=seq2name)
det_fate = det_fate.sort_index(ascending=False).rename(index=seq2name)

ee = model_groups_data['ee']
biomass = model_groups_data['biomass']
ge = model_groups_data['gross_efficiency']  # te
gs = model_groups_data['prop_unassimilated_food']  # te
p = model_groups_data['biomass'] * model_groups_data['pb']
q = model_groups_data['biomass'] * model_groups_data['qb']
p_times_ee = p * ee
predation = Z.sum(axis=0)
inner_q = Z.sum(axis=1)
M0 = model_groups_data['biomass'] * model_groups_data['pb'] * (1-model_groups_data['ee'])
# M0 = model_groups_data['biomass'] * model_groups_data['pb']
export = model_groups_data['export']
growth = model_groups_data['biomass_accum']
net_migration = + model_groups_data['emigration'] - model_groups_data['immigration']
egestion = model_groups_data["biomass"] * model_groups_data["qb"] * (model_groups_data["prop_unassimilated_food"]) * (DC * (1-det_fate)).sum(axis=1)
respiration = model_groups_data['respiration']

# q[non_det] = inner_q[non_det]
# growth[det] + predation[det] = inner_q[det]

df = pd.DataFrame({
    # 'biomass': biomass,
    'q': q,
    'p': p,
    'test': q - (p + egestion + respiration),  # should be 0
    'test3': p - (predation + M0 + growth + net_migration + export),  # should be 0 (definition of P)
    'test4': p_times_ee - (export + predation +  growth + net_migration),  # should be 0 (definition of EE)
    'predation': predation,
    'ee': ee,
    'ge': ge,
    'te': p / q,
    'TE': (predation + M0 + net_growth + export) / inner_q,
    'gs': gs,
    'M0': M0,
    'egestion': egestion,
    'respiration': respiration,
    'export': export,
    'growth': net_growth,
    'ee': ee,
    'p_times_ee': p_times_ee,
    'inner_q': inner_q,
})
df.round(decimals=5)
# model_groups_data.round(decimals=5)

,q,p,test,test3,test4,predation,ee,ge,te,TE,gs,M0,egestion,respiration,export,growth,p_times_ee,inner_q
Detritus,0.00000,0.00000,0.00000,-17034.81551,-17034.81551,305.33891,0.01792,0.00000,NaN,inf,0.0,0.00000,0.00000,0.00000,0.00000,16729.4766,0.00000,0.00000
Phytoplankton,0.00000,919.34252,-919.34252,-0.00005,-0.00005,367.73706,0.40000,0.00000,inf,inf,0.0,551.60551,0.00000,0.00000,0.00000,0.0000,367.73701,0.00000
Benthic producers,0.00000,16324.55000,-16324.55000,0.02655,0.02655,129.37651,0.00793,0.00000,inf,inf,0.0,16195.14694,0.00000,0.00000,0.00000,0.0000,129.40306,0.00000
Zooplankton,363.86023,107.15625,0.00002,0.00001,0.00001,101.79843,0.95000,0.29450,0.29450,0.29450,0.4,5.35781,145.54409,111.15987,0.00000,0.0000,101.79844,363.86023
Other Fish,27.33101,5.46620,-0.00000,0.00000,0.00000,5.19289,0.95000,0.20000,0.20000,0.20000,0.2,0.27331,5.46620,16.39861,0.00000,0.0000,5.19289,27.33101
Benthos,431.25503,38.81295,60.37570,-0.00000,-0.00000,19.40648,0.50000,0.09000,0.09000,0.09000,0.2,19.40648,25.87530,306.19107,0.00000,0.0000,19.40648,431.25503
Molluscs,7.39004,1.10851,0.34742,-0.00000,-0.00000,0.83138,0.75000,0.15000,0.15000,0.15000,0.2,0.27713,1.13059,4.80352,0.00000,0.0000,0.83138,7.39004
Northern Shrimp,9.17355,1.83471,0.18347,0.00000,0.00000,1.74297,0.95000,0.20000,0.20000,0.20000,0.2,0.09174,1.65124,5.50413,0.00000,0.0000,1.74297,9.17355
Nephrops,0.43538,0.08708,0.01306,-0.00000,-0.00000,0.08268,0.95000,0.20000,0.20000,0.20000,0.2,0.00435,0.07402,0.26123,0.00004,0.0000,0.08272,0.43538
Other Pelagics,30.56259,4.58439,-0.00000,0.00000,0.00000,4.35326,0.95000,0.15000,0.15000,0.15000,0.2,0.22922,6.11252,19.86568,0.00191,0.0000,4.35517,30.56259


In [86]:
df2 = pd.DataFrame({
    'test': q - (p + egestion + respiration),  # should be 0
    'test_r': (q - (p + egestion + respiration)) / q,  # small relative error
    'test2': q - (predation + M0 + net_growth + egestion + respiration),
    'test2_r': (q - (predation + M0 + net_growth + egestion + respiration)) / q,  # mostly small relative error
    'test3': p - (predation + M0 + net_growth + export),
    'test3_r': (p - (predation + M0 + net_growth + export)) / p,  # high relative error
    'test4': p_times_ee - predation,  # should be 0
    'test4_r': (p_times_ee - predation) / p_times_ee,  # high should be 0
})
df2.round(decimals=5)

,test,test_r,test2,test2_r,test3,test3_r,test4,test4_r
group_seq,,,,,,,,
Detritus,0.00000,NaN,-17034.81551,-inf,-17034.81551,-inf,-305.33891,-inf
Phytoplankton,-919.34252,-inf,-919.34257,-inf,-0.00005,-0.0,-0.00005,-0.00000
Benthic producers,-16324.55000,-inf,-16324.52345,-inf,0.02655,0.0,0.02655,0.00021
Zooplankton,0.00002,0.00000,0.00003,0.00000,0.00001,0.0,0.00001,0.00000
Other Fish,-0.00000,-0.00000,-0.00000,-0.00000,0.00000,0.0,0.00000,0.00000
Benthos,60.37570,0.14000,60.37570,0.14000,-0.00000,-0.0,-0.00000,-0.00000
Molluscs,0.34742,0.04701,0.34742,0.04701,-0.00000,-0.0,-0.00000,-0.00000
Northern Shrimp,0.18347,0.02000,0.18347,0.02000,0.00000,0.0,0.00000,0.00000
Nephrops,0.01306,0.03000,0.01310,0.03010,-0.00000,-0.0,0.00004,0.00051


In [2]:
Z

,1,2,3,4,5,6,7,8,9,10,...,14,15,16,17,18,19,20,21,23,24
1,0.000789,0.003794,0.003794,0.003794,0.015531,0.015531,0.015531,0.015531,0.047767,0.015531,...,0.018599,0.015531,0.010999,0.010999,0.253672,0.010999,0.015531,0.001897,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.082644,0.082644,0.082644,0.082644,0.165287,0.082644,...,0.454540,0.082644,0.057851,0.110192,0.165287,0.110192,0.082644,9.256093,0.000000,0.000000
3,0.000000,0.000603,0.001822,0.005033,0.085104,0.085104,0.060227,0.085104,0.013926,0.085104,...,0.222818,0.085104,0.004040,0.190324,0.236745,0.190324,0.085104,0.153188,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.913767,0.032370,0.000000,0.000000,0.000000,0.685326,0.091377,0.161852,0.000000,0.138730
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.032795,0.021604,0.032619,0.064685,0.033813,...,1.825602,0.351594,0.000000,0.750524,0.000000,0.743758,0.574728,1.386099,0.000000,0.000000
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.201517,0.000000,0.550814,0.094041,0.362731,0.134345,0.000000
7,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.005909,0.000000,...,0.196981,0.000000,0.000000,0.068943,0.000000,1.047940,0.025608,0.610642,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.002040,0.020402,0.000000,0.020402,0.000000,...,0.591649,0.002040,0.000000,0.000000,0.000000,0.163214,0.530444,0.701818,0.000000,0.002040
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.030793,0.000000,0.000000,0.000000,0.000000,...,0.184757,0.123171,0.000000,0.080061,0.000000,0.923785,0.246343,4.569657,0.000000,0.000000
10,0.000000,0.000000,0.000000,0.000000,0.089403,0.059320,0.000000,0.000000,0.260961,0.096652,...,0.471180,0.118640,0.000000,0.079094,0.025705,0.033615,0.158187,0.001977,0.000000,0.000000


# Northen Gulf of Saint Lawrence

In [68]:
model_number = 116  # Northern Gulf Od Saint Lawrence #1
# model_number = 462  # Northern Gulf Od Saint Lawrence #2

seq2name = get_seq2name(model_number)
model_groups_data = get_model_groups_data(model_number)
DC, det_fate = get_DC(model_number)

# flow to det:
# flow_from_food = model_groups_data["biomass"] * model_groups_data["qb"] * model_groups_data["prop_unassimilated_food"] * (DC * (1-det_fate)).sum(axis=1)
flow_from_food = model_groups_data["biomass"] * model_groups_data["qb"] * model_groups_data["prop_unassimilated_food"]
flow_from_bodies = model_groups_data["biomass"] * model_groups_data["M0b"]
model_groups_data['M0'] = flow_from_bodies
model_groups_data['flow_to_det'] = flow_from_food + flow_from_bodies

net_growth = model_groups_data['export'] + model_groups_data['biomass_accum'] + model_groups_data['emigration']
model_groups_data['net_growth'] = net_growth
respiration = model_groups_data['respiration']
production = model_groups_data['p']

# model_groups_data['p'].sort_index()
model_groups_data.sort_index()
# resipiration2 = model_groups_data['q'] * (1-model_groups_data['prop_unassimilated_food']) - model_groups_data['p']
# model_groups_data['r2'] = resipiration2
# model_groups_data[['r2', 'respiration']]

# get Z:
# Z = get_Z(model_number, use_det_fate=False)
# Z.rename(columns=seq2name).T  # should be like in the article

# # combine PP to single row:
# PP_seq_list = model_groups_data.index[model_groups_data['trophic_info'] == 'PP'].values
# PP_seq = PP_seq_list[-1]
# if len(PP_seq_list) > 1:
#     Z, net_growth, respiration, production = merge_pp([Z, net_growth, respiration, production], PP_seq_list)

# DET_seq = model_groups_data[model_groups_data['trophic_info'] == 'DET'].index.values[0]

# # combine part of DET that is PP into PP row:
# Z_without_DET = Z.copy()
# percent_of_det_that_is_PP = Z_without_DET.loc[DET_seq, PP_seq] / Z_without_DET.loc[DET_seq, :].sum()  # 98%
# Z_without_DET.loc[:, PP_seq] += percent_of_det_that_is_PP * Z_without_DET.loc[:, DET_seq]
# Z_without_DET = Z_without_DET.drop(index=DET_seq, columns=DET_seq)

# # production of living compartments:
# P = production.add(net_growth, fill_value=0)
# if isinstance(P, pd.DataFrame):
#     P = P.sum(axis=1)
# non_PP = model_groups_data['trophic_info'] == 'Regular'
# ee = model_groups_data[non_PP]["ee"]
# export = -model_groups_data[non_PP]["export"]
# P[non_PP] = P[non_PP].add(export).mul(ee)
# P = P.drop(index=DET_seq).sort_index(ascending=False).rename(index=seq2name)

# # production-normalized transaction matrix:
# A = Z_without_DET.T.sort_index(ascending=False).sort_index(axis=1, ascending=False).rename(index=seq2name, columns=seq2name) / P

# # production requirement matrix:
# seq_to_drop = P.index[P == 0]
# A = A.drop(columns=seq_to_drop, index=seq_to_drop)
# L = pd.DataFrame(np.linalg.inv((np.identity(A.shape[0]) - A)), index=A.index, columns=A.columns)

# L

,group_name,trophic_info,export,tl,biomass,pb,qb,p,q,respiration,...,prop_unassimilated_food,ee,M0b,flow_to_det,biomass_accum,biomass_accum_rate,emigration,immigration,M0,net_growth
group_seq,,,,,,,,,,,,,,,,,,,,,
1,Cetacea,Regular,0.000400,4.093985,0.120000,0.072643,7.265048,0.008717,0.871806,0.775908,...,0.100,0.045886,0.069310,0.095498,0.0,0.0,0.0,0.0,0.008317,0.000400
2,Harp seals,Regular,0.004200,4.247342,0.042961,0.102909,10.291240,0.004421,0.442121,0.305063,...,0.300,0.950000,0.005145,0.132857,0.0,0.0,0.0,0.0,0.000221,0.004200
3,Hooded seals,Regular,0.000010,4.857662,0.003292,0.053410,9.658946,0.000176,0.031800,0.022084,...,0.300,0.056870,0.050372,0.009706,0.0,0.0,0.0,0.0,0.000166,0.000010
4,Grey seals,Regular,0.000700,4.516014,0.013000,0.109341,10.933355,0.001421,0.142134,0.098072,...,0.300,0.492459,0.055495,0.043362,0.0,0.0,0.0,0.0,0.000721,0.000700
5,Harbour seals,Regular,0.000000,4.158625,0.002000,0.070937,7.091081,0.000142,0.014182,0.009786,...,0.300,0.000000,0.070937,0.004397,0.0,0.0,0.0,0.0,0.000142,0.000000
6,Seabirds,Regular,0.000300,4.141750,0.002600,0.401664,42.050556,0.001044,0.109331,0.075488,...,0.300,0.287266,0.286280,0.033544,0.0,0.0,0.0,0.0,0.000744,0.000300
7,Large cod,Regular,0.694000,4.229020,2.009000,0.826202,2.757616,1.659840,5.540050,2.218195,...,0.300,0.449252,0.455029,2.576169,0.0,0.0,0.0,0.0,0.914154,0.694000
8,Small cod,Regular,0.000900,3.595177,0.820000,0.819911,4.032117,0.672327,3.306336,1.642108,...,0.300,0.809393,0.156280,1.120051,0.0,0.0,0.0,0.0,0.128150,0.000900
9,Large Green. halibut,Regular,0.039500,4.007891,0.310000,0.159938,1.599346,0.049581,0.495797,0.297478,...,0.300,0.951906,0.007692,0.151124,0.0,0.0,0.0,0.0,0.002385,0.039500
